# 1) My Lane as an ML Task
* **Lane:** Lane 2: Refresh / Content Opportunity Scoring
* **ML Task Type:** Supervised Learning — Binary Classification (predicting whether a page will decline) paired with Learning to Rank (prioritizing the risk queue).

# 2) Target or Proxy
Our target label is **`is_declining_label`** (a proxy built from `trend_direction == 'down'`). It is a proxy because it indicates active traffic loss in the current window, suggesting the page has stale content and needs a refresh to recover.

# 3) Success Metric
* **Primary Metric:** **Precision@50** (the proportion of the top 50 flagged pages that are actually in decline). This matches the human reviewer's weekly capacity of checking 50 pages.
* **Secondary Metric:** **ROC AUC** to measure overall classification ranking quality.

In [1]:
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Define the unit of analysis (One row = One unique content page URL)
unit_of_analysis = df[['content_id', 'client_id', 'impressions_90d', 'trend_direction']].drop_duplicates().head(5)
print("Unit of Analysis (One row = One unique content page):")
display(unit_of_analysis)

# Sketch the target column
df['target_is_declining'] = (df['trend_direction'] == 'down').astype(int)
print("\nTarget Column Sketch (1 = Declining, 0 = Stable/Growing):")
display(df[['content_id', 'trend_direction', 'target_is_declining']].head(5))

Unit of Analysis (One row = One unique content page):


,content_id,client_id,impressions_90d,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,down
1,content_a1fb4e703a9e,client_4e07408562,15320,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,down
3,content_331d6c4de07b,client_19581e27de,11751,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,down



Target Column Sketch (1 = Declining, 0 = Stable/Growing):


,content_id,trend_direction,target_is_declining
0,content_304f48230142,down,1
1,content_a1fb4e703a9e,down,1
2,content_9aa793d4d895,down,1
3,content_331d6c4de07b,stable,0
4,content_d99b7a2d90ca,down,1


# 5) Why ML Beats a Fixed Rule
A human-written rule like "if age > 180 days, rewrite it" is too rigid and misses high-potential traffic pages that decay early. ML beats fixed rules by dynamically weighting interactions between 40+ features (like engagement rate, position tiers, and search volume changes) to rank the highest-opportunity pages first.